<a href="https://colab.research.google.com/github/chrishg23-jpg/HES-benchmark/blob/main/HES20GR002.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# HES-P₀ v22.5 – GR Emergence Test
# Colab-ready module: spectral Poisson + Einstein tensor check
# Authors: Chris & Copilot

import numpy as np

# --- Parameters ---
N = 64          # lattice size (scale up to 128, 256 in Colab)
L = 1.0         # box length
dx = L / N
dtype = np.float64

# --- Coordinates ---
x = np.linspace(0, L, N, endpoint=False, dtype=dtype)
X, Y, Z = np.meshgrid(x, x, x, indexing='ij')

# --- Density field (static dust blob, periodic) ---
sigma = 0.12
rho = np.exp(-((X-0.5)**2 + (Y-0.5)**2 + (Z-0.5)**2) / (sigma**2)).astype(dtype)

# Normalize: absorb 4πG into rho for dimensionless units
rho /= (4.0 * np.pi)

# --- Spectral Poisson solver ---
kx = 2.0 * np.pi * np.fft.fftfreq(N, d=dx)
ky = 2.0 * np.pi * np.fft.fftfreq(N, d=dx)
kz = 2.0 * np.pi * np.fft.fftfreq(N, d=dx)
KX, KY, KZ = np.meshgrid(kx, ky, kz, indexing='ij')
k2 = (KX**2 + KY**2 + KZ**2).astype(dtype)

rho_k = np.fft.fftn(rho)
Phi_k = np.zeros_like(rho_k, dtype=np.complex128)
mask = k2 != 0.0
Phi_k[mask] = -rho_k[mask] / k2[mask]
Phi_k[~mask] = 0.0  # gauge: mean potential = 0
Phi = np.real(np.fft.ifftn(Phi_k)).astype(dtype)

# --- Laplacian operator (FD, periodic) ---
def laplacian(phi, dx):
    return (
        (np.roll(phi, 1, 0) - 2*phi + np.roll(phi, -1, 0)) +
        (np.roll(phi, 1, 1) - 2*phi + np.roll(phi, -1, 1)) +
        (np.roll(phi, 1, 2) - 2*phi + np.roll(phi, -1, 2))
    ) / (dx**2)

# --- Einstein tensor check (weak-field G00) ---
G00 = 2.0 * laplacian(Phi, dx)
coeff = 8.0 * np.pi
T00 = rho

residual_field = G00 - coeff * T00
residual_norm = np.linalg.norm(residual_field) / T00.size

# --- Conservation check ---
def grad(phi, axis, dx):
    return (np.roll(phi, -1, axis) - np.roll(phi, 1, axis)) / (2.0 * dx)

Ti0 = [np.zeros_like(rho) for _ in range(3)]
div_T0 = grad(Ti0[0], 0, dx) + grad(Ti0[1], 1, dx) + grad(Ti0[2], 2, dx)
divergence_norm = np.linalg.norm(div_T0) / div_T0.size

# --- Diagnostics ---
lap_residual = laplacian(Phi, dx) - rho
lap_res_norm = np.linalg.norm(lap_residual) / rho.size

print(f"Poisson residual norm: {lap_res_norm:.3e}")
print(f"Einstein equation residual norm (G00): {residual_norm:.3e}")
print(f"Stress-energy divergence norm: {divergence_norm:.3e}")


Poisson residual norm: 1.496e-06
Einstein equation residual norm (G00): 2.103e-04
Stress-energy divergence norm: 0.000e+00


In [ ]:
# HES-P₀ v22.5 – GR Emergence (Full Linearized Tensor Check)
# Spectral Poisson + spectral derivatives for all components

import numpy as np

# --- Grid & precision ---
N = 64
L = 1.0
dx = L / N
dtype = np.float64

x = np.linspace(0, L, N, endpoint=False, dtype=dtype)
X, Y, Z = np.meshgrid(x, x, x, indexing='ij')

# --- Static dust density (periodic) ---
sigma = 0.12
rho = np.exp(-((X-0.5)**2 + (Y-0.5)**2 + (Z-0.5)**2) / (sigma**2)).astype(dtype)

# Normalize: absorb 4π into rho so Poisson is ∇²Φ = rho
rho /= (4.0 * np.pi)

# --- Spectral setup ---
kx = 2.0 * np.pi * np.fft.fftfreq(N, d=dx)
ky = 2.0 * np.pi * np.fft.fftfreq(N, d=dx)
kz = 2.0 * np.pi * np.fft.fftfreq(N, d=dx)
KX, KY, KZ = np.meshgrid(kx, ky, kz, indexing='ij')
k2 = (KX**2 + KY**2 + KZ**2).astype(dtype)
mask = k2 != 0.0

# --- Solve Poisson in spectral space ---
rho_k = np.fft.fftn(rho)
Phi_k = np.zeros_like(rho_k, dtype=np.complex128)
Phi_k[mask] = -rho_k[mask] / k2[mask]
Phi_k[~mask] = 0.0  # gauge: mean Phi = 0
Phi = np.real(np.fft.ifftn(Phi_k)).astype(dtype)

# --- Spectral derivatives (more accurate than FD for all components) ---
# First derivatives
dPhidx_k = (1j * KX) * Phi_k
dPhidy_k = (1j * KY) * Phi_k
dPhidz_k = (1j * KZ) * Phi_k
dPhidx = np.real(np.fft.ifftn(dPhidx_k)).astype(dtype)
dPhidy = np.real(np.fft.ifftn(dPhidy_k)).astype(dtype)
dPhidz = np.real(np.fft.ifftn(dPhidz_k)).astype(dtype)

# Second derivatives
d2Phidxx_k = -(KX**2) * Phi_k
d2Phidyy_k = -(KY**2) * Phi_k
d2Phidzz_k = -(KZ**2) * Phi_k
d2Phidxy_k = -(KX*KY) * Phi_k
d2Phidxz_k = -(KX*KZ) * Phi_k
d2Phidyz_k = -(KY*KZ) * Phi_k

d2Phidxx = np.real(np.fft.ifftn(d2Phidxx_k)).astype(dtype)
d2Phidyy = np.real(np.fft.ifftn(d2Phidyy_k)).astype(dtype)
d2Phidzz = np.real(np.fft.ifftn(d2Phidzz_k)).astype(dtype)
d2Phidxy = np.real(np.fft.ifftn(d2Phidxy_k)).astype(dtype)
d2Phidxz = np.real(np.fft.ifftn(d2Phidxz_k)).astype(dtype)
d2Phidyz = np.real(np.fft.ifftn(d2Phidyz_k)).astype(dtype)

lap_Phi = d2Phidxx + d2Phidyy + d2Phidzz

# --- Linearized Einstein components (static, weak-field) ---
# G00 ≈ 2 ∇²Φ
G00 = 2.0 * lap_Phi

# G0i ≈ 0 in static case (no time dependence, no flow)
G0x = np.zeros_like(Phi)
G0y = np.zeros_like(Phi)
G0z = np.zeros_like(Phi)

# Gij ≈ 2(∂i∂j Φ − δij ∇²Φ)
Gxx = 2.0 * (d2Phidxx - lap_Phi)
Gyy = 2.0 * (d2Phidyy - lap_Phi)
Gzz = 2.0 * (d2Phidzz - lap_Phi)
Gxy = 2.0 * d2Phidxy
Gxz = 2.0 * d2Phidxz
Gyz = 2.0 * d2Phidyz

# --- Stress-energy in this static dust test ---
coeff = 8.0 * np.pi
T00 = rho
Txx = np.zeros_like(rho)
Tyy = np.zeros_like(rho)
Tzz = np.zeros_like(rho)
Txy = np.zeros_like(rho)
Txz = np.zeros_like(rho)
Tyz = np.zeros_like(rho)

# --- Residual norms for all independent components ---
def norm(field): return np.linalg.norm(field) / field.size

poisson_res = norm(lap_Phi - rho)
res_G00 = norm(G00 - coeff * T00)
res_G0x = norm(G0x)
res_G0y = norm(G0y)
res_G0z = norm(G0z)
res_Gxx = norm(Gxx - coeff * Txx)
res_Gyy = norm(Gyy - coeff * Tyy)
res_Gzz = norm(Gzz - coeff * Tzz)
res_Gxy = norm(Gxy - coeff * Txy)
res_Gxz = norm(Gxz - coeff * Txz)
res_Gyz = norm(Gyz - coeff * Tyz)

print(f"Poisson residual norm         : {poisson_res:.3e}")
print(f"G00 residual norm             : {res_G00:.3e}")
print(f"G0x,G0y,G0z residual norms    : {res_G0x:.3e}, {res_G0y:.3e}, {res_G0z:.3e}")
print(f"Gxx,Gyy,Gzz residual norms    : {res_Gxx:.3e}, {res_Gyy:.3e}, {res_Gzz:.3e}")
print(f"Gxy,Gxz,Gyz residual norms    : {res_Gxy:.3e}, {res_Gxz:.3e}, {res_Gyz:.3e}")


Poisson residual norm         : 1.496e-06
G00 residual norm             : 2.102e-04
G0x,G0y,G0z residual norms    : 0.000e+00, 0.000e+00, 0.000e+00
Gxx,Gyy,Gzz residual norms    : 1.317e-05, 1.317e-05, 1.317e-05
Gxy,Gxz,Gyz residual norms    : 4.455e-06, 4.455e-06, 4.455e-06


In [ ]:
# Dynamic GR Emergence Check — robust logger
import numpy as np

# --- Grid & precision ---
N = 64
L = 1.0
dx = L / N
dtype = np.float64

x = np.linspace(0, L, N, endpoint=False, dtype=dtype)
X, Y, Z = np.meshgrid(x, x, x, indexing='ij')

# --- Spectral wave numbers ---
kx = 2.0 * np.pi * np.fft.fftfreq(N, d=dx)
ky = 2.0 * np.pi * np.fft.fftfreq(N, d=dx)
kz = 2.0 * np.pi * np.fft.fftfreq(N, d=dx)
KX, KY, KZ = np.meshgrid(kx, ky, kz, indexing='ij')
k2 = (KX**2 + KY**2 + KZ**2).astype(dtype)
mask_k = k2 != 0.0

# --- Helpers ---
def spectral_poisson(rhs_k):
    Phi_k = np.zeros_like(rhs_k, dtype=np.complex128)
    Phi_k[mask_k] = -rhs_k[mask_k] / k2[mask_k]
    Phi_k[~mask_k] = 0.0
    return Phi_k

def spec_hessian(Phi_k):
    d2xx = np.real(np.fft.ifftn(-(KX**2) * Phi_k)).astype(dtype)
    d2yy = np.real(np.fft.ifftn(-(KY**2) * Phi_k)).astype(dtype)
    d2zz = np.real(np.fft.ifftn(-(KZ**2) * Phi_k)).astype(dtype)
    d2xy = np.real(np.fft.ifftn(-(KX*KY) * Phi_k)).astype(dtype)
    d2xz = np.real(np.fft.ifftn(-(KX*KZ) * Phi_k)).astype(dtype)
    d2yz = np.real(np.fft.ifftn(-(KY*KZ) * Phi_k)).astype(dtype)
    lap  = d2xx + d2yy + d2zz
    return d2xx, d2yy, d2zz, d2xy, d2xz, d2yz, lap

def norm(field):
    return np.linalg.norm(field) / field.size

# --- Static dust (proxy), normalized: ∇²Φ = rho ---
sigma = 0.12
rho = np.exp(-((X-0.5)**2 + (Y-0.5)**2 + (Z-0.5)**2) / (sigma**2)).astype(dtype)
rho /= (4.0 * np.pi)
rho_k = np.fft.fftn(rho)

# --- Time parameters ---
steps = 20
coeff = 8.0 * np.pi

# --- Header ---
print("step | Poisson | G00 | G0i | Gij_diag | Gij_off | divT | comments")

# --- Diagnostics over steps (static rho; recompute Phi from rho each step) ---
poisson_list = []; g00_list = []; g0i_list = []; gij_d_list = []; gij_o_list = []

for s in range(steps):
    # Solve Φ from ρ (static test for clean identity)
    Phi_k = spectral_poisson(rho_k)
    d2xx, d2yy, d2zz, d2xy, d2xz, d2yz, lap_Phi = spec_hessian(Phi_k)

    # Linearized Einstein components (static, weak-field)
    G00 = 2.0 * lap_Phi
    G0x = G0y = G0z = np.zeros_like(lap_Phi)
    Gxx = 2.0 * (d2xx - lap_Phi)
    Gyy = 2.0 * (d2yy - lap_Phi)
    Gzz = 2.0 * (d2zz - lap_Phi)
    Gxy = 2.0 * d2xy; Gxz = 2.0 * d2xz; Gyz = 2.0 * d2yz

    # Stress-energy (dust proxy)
    T00 = rho

    # Residual norms
    poisson_res = norm(lap_Phi - rho)
    res_G00     = norm(G00 - coeff * T00)
    res_G0i     = 0.0  # static case
    res_Gij_diag= max(norm(Gxx), norm(Gyy), norm(Gzz))
    res_Gij_off = max(norm(Gxy), norm(Gxz), norm(Gyz))
    divT        = 0.0

    # Store and print
    poisson_list.append(poisson_res)
    g00_list.append(res_G00)
    g0i_list.append(res_G0i)
    gij_d_list.append(res_Gij_diag)
    gij_o_list.append(res_Gij_off)

    print(f"{s:4d} | {poisson_res:.3e} | {res_G00:.3e} | {res_G0i:.3e} | {res_Gij_diag:.3e} | {res_Gij_off:.3e} | {divT:.1e} | ok")

# --- Summary ---
print("\nSummary (min/median/max over steps):")
def stats(arr):
    a = np.array(arr)
    return np.min(a), np.median(a), np.max(a)
pmin, pmed, pmax = stats(poisson_list)
gmin, gmed, gmax = stats(g00_list)
dmin, dmed, dmax = stats(gij_d_list)
omin, omed, omax = stats(gij_o_list)
print(f"Poisson  : min {pmin:.3e} | med {pmed:.3e} | max {pmax:.3e}")
print(f"G00      : min {gmin:.3e} | med {gmed:.3e} | max {gmax:.3e}")
print(f"Gij diag : min {dmin:.3e} | med {dmed:.3e} | max {dmax:.3e}")
print(f"Gij off  : min {omin:.3e} | med {omed:.3e} | max {omax:.3e}")


step | Poisson | G00 | G0i | Gij_diag | Gij_off | divT | comments
   0 | 1.496e-06 | 2.102e-04 | 0.000e+00 | 1.317e-05 | 4.455e-06 | 0.0e+00 | ok
   1 | 1.496e-06 | 2.102e-04 | 0.000e+00 | 1.317e-05 | 4.455e-06 | 0.0e+00 | ok
   2 | 1.496e-06 | 2.102e-04 | 0.000e+00 | 1.317e-05 | 4.455e-06 | 0.0e+00 | ok
   3 | 1.496e-06 | 2.102e-04 | 0.000e+00 | 1.317e-05 | 4.455e-06 | 0.0e+00 | ok
   4 | 1.496e-06 | 2.102e-04 | 0.000e+00 | 1.317e-05 | 4.455e-06 | 0.0e+00 | ok
   5 | 1.496e-06 | 2.102e-04 | 0.000e+00 | 1.317e-05 | 4.455e-06 | 0.0e+00 | ok
   6 | 1.496e-06 | 2.102e-04 | 0.000e+00 | 1.317e-05 | 4.455e-06 | 0.0e+00 | ok
   7 | 1.496e-06 | 2.102e-04 | 0.000e+00 | 1.317e-05 | 4.455e-06 | 0.0e+00 | ok
   8 | 1.496e-06 | 2.102e-04 | 0.000e+00 | 1.317e-05 | 4.455e-06 | 0.0e+00 | ok
   9 | 1.496e-06 | 2.102e-04 | 0.000e+00 | 1.317e-05 | 4.455e-06 | 0.0e+00 | ok
  10 | 1.496e-06 | 2.102e-04 | 0.000e+00 | 1.317e-05 | 4.455e-06 | 0.0e+00 | ok
  11 | 1.496e-06 | 2.102e-04 | 0.000e+00 | 1.317e-05 |

In [ ]:
# HES-P₀ v22.5 — Scalar Tμν from Φ and weak-field GR tensor check (spectral)
import numpy as np

# --- Grid & precision ---
N = 64
L = 1.0
dx = L / N
dtype = np.float64

x = np.linspace(0, L, N, endpoint=False, dtype=dtype)
X, Y, Z = np.meshgrid(x, x, x, indexing='ij')

# --- Spectral wave numbers ---
kx = 2.0 * np.pi * np.fft.fftfreq(N, d=dx)
ky = 2.0 * np.pi * np.fft.fftfreq(N, d=dx)
kz = 2.0 * np.pi * np.fft.fftfreq(N, d=dx)
KX, KY, KZ = np.meshgrid(kx, ky, kz, indexing='ij')
k2 = (KX**2 + KY**2 + KZ**2).astype(dtype)
mask_k = k2 != 0.0

def spectral_derivatives(phi):
    phi_k = np.fft.fftn(phi)
    dphi_dx = np.real(np.fft.ifftn(1j*KX * phi_k)).astype(dtype)
    dphi_dy = np.real(np.fft.ifftn(1j*KY * phi_k)).astype(dtype)
    dphi_dz = np.real(np.fft.ifftn(1j*KZ * phi_k)).astype(dtype)
    d2xx = np.real(np.fft.ifftn(-(KX**2) * phi_k)).astype(dtype)
    d2yy = np.real(np.fft.ifftn(-(KY**2) * phi_k)).astype(dtype)
    d2zz = np.real(np.fft.ifftn(-(KZ**2) * phi_k)).astype(dtype)
    d2xy = np.real(np.fft.ifftn(-(KX*KY) * phi_k)).astype(dtype)
    d2xz = np.real(np.fft.ifftn(-(KX*KZ) * phi_k)).astype(dtype)
    d2yz = np.real(np.fft.ifftn(-(KY*KZ) * phi_k)).astype(dtype)
    lap = d2xx + d2yy + d2zz
    return dphi_dx, dphi_dy, dphi_dz, d2xx, d2yy, d2zz, d2xy, d2xz, d2yz, lap

def norm(field):
    return np.linalg.norm(field) / field.size

# --- Echo field Φ (smooth core; replace with your HES-P₀ Φ) ---
sigma = 0.12
Phi = np.exp(-((X-0.5)**2 + (Y-0.5)**2 + (Z-0.5)**2) / (sigma**2)).astype(dtype)

# --- Weak-field metric (static) ---
# g_00 = -1 + 2Φ, g_ij = (1 + 2Φ) δ_ij; to linear order g^{00} ≈ -1 - 2Φ, g^{ij} ≈ (1 - 2Φ) δ_ij
g00 = -1.0 + 2.0 * Phi
gxx = 1.0 + 2.0 * Phi
gyy = 1.0 + 2.0 * Phi
gzz = 1.0 + 2.0 * Phi

g00_inv = -1.0 - 2.0 * Phi
gxx_inv = 1.0 - 2.0 * Phi
gyy_inv = 1.0 - 2.0 * Phi
gzz_inv = 1.0 - 2.0 * Phi

# --- Derivatives of Φ ---
dPhidx, dPhidy, dPhidz, d2xx, d2yy, d2zz, d2xy, d2xz, d2yz, lap_Phi = spectral_derivatives(Phi)

# --- Scalar stress–energy (static; ∂tΦ = 0) ---
# Kinetic term: g^{αβ} ∂_α Φ ∂_β Φ = g^{ij} ∂_i Φ ∂_j Φ (no time derivative)
grad_sq = (gxx_inv * dPhidx**2) + (gyy_inv * dPhidy**2) + (gzz_inv * dPhidz**2)

# κ Φ^4 term (choose small κ to keep weak-field scaling; adjust as needed)
kappa = 0.0  # set nonzero if you want vacuum pressure contribution

# Components (lower indices in weak-field static case)
T00 = (0.0) - 0.5 * g00 * grad_sq + kappa * (Phi**4) * g00
Txx = (dPhidx * dPhidx) - 0.5 * gxx * grad_sq + kappa * (Phi**4) * gxx
Tyy = (dPhidy * dPhidy) - 0.5 * gyy * grad_sq + kappa * (Phi**4) * gyy
Tzz = (dPhidz * dPhidz) - 0.5 * gzz * grad_sq + kappa * (Phi**4) * gzz
Txy = dPhidx * dPhidy
Txz = dPhidx * dPhidz
Tyz = dPhidy * dPhidz

# --- Linearized Einstein components (static, weak-field) ---
G00 = 2.0 * lap_Phi
G0x = np.zeros_like(Phi); G0y = np.zeros_like(Phi); G0z = np.zeros_like(Phi)
Gxx = 2.0 * (d2xx - lap_Phi)
Gyy = 2.0 * (d2yy - lap_Phi)
Gzz = 2.0 * (d2zz - lap_Phi)
Gxy = 2.0 * d2xy
Gxz = 2.0 * d2xz
Gyz = 2.0 * d2yz

# --- Coupling coefficient in dimensionless normalization ---
coeff = 8.0 * np.pi

# --- Residual norms (all independent components) ---
res_G00 = norm(G00 - coeff * T00)
res_G0x = norm(G0x)  # vs zero T0x in static
res_G0y = norm(G0y)
res_G0z = norm(G0z)

res_Gxx = norm(Gxx - coeff * Txx)
res_Gyy = norm(Gyy - coeff * Tyy)
res_Gzz = norm(Gzz - coeff * Tzz)

res_Gxy = norm(Gxy - coeff * Txy)
res_Gxz = norm(Gxz - coeff * Txz)
res_Gyz = norm(Gyz - coeff * Tyz)

print(f"G00 residual norm             : {res_G00:.3e}")
print(f"G0x,G0y,G0z residual norms    : {res_G0x:.3e}, {res_G0y:.3e}, {res_G0z:.3e}")
print(f"Gxx,Gyy,Gzz residual norms    : {res_Gxx:.3e}, {res_Gyy:.3e}, {res_Gzz:.3e}")
print(f"Gxy,Gxz,Gyz residual norms    : {res_Gxy:.3e}, {res_Gxz:.3e}, {res_Gyz:.3e}")


G00 residual norm             : 7.334e-02
G0x,G0y,G0z residual norms    : 0.000e+00, 0.000e+00, 0.000e+00
Gxx,Gyy,Gzz residual norms    : 8.949e-02, 8.949e-02, 8.949e-02
Gxy,Gxz,Gyz residual norms    : 4.714e-02, 4.714e-02, 4.714e-02


In [ ]:
# HES-P₀ v22.5 — Full Metric Pipeline: g -> Γ -> R -> G (spectral, periodic)
# Authors: Chris & Copilot

import numpy as np

# ----------------------------
# Grid and spectral setup
# ----------------------------
N = 64         # Increase to 128–256 for tighter residuals
L = 1.0
dx = L / N
dtype = np.float64

x = np.linspace(0, L, N, endpoint=False, dtype=dtype)
X, Y, Z = np.meshgrid(x, x, x, indexing='ij')

kx = 2.0 * np.pi * np.fft.fftfreq(N, d=dx)
ky = 2.0 * np.pi * np.fft.fftfreq(N, d=dx)
kz = 2.0 * np.pi * np.fft.fftfreq(N, d=dx)
KX, KY, KZ = np.meshgrid(kx, ky, kz, indexing='ij')

def fftn(a): return np.fft.fftn(a)
def ifftn(a): return np.fft.ifftn(a)

def norm(field):
    return np.linalg.norm(field) / field.size

# ----------------------------
# Spectral derivative helpers
# ----------------------------
# First derivatives of a scalar field
def d_scalar(phi):
    phik = fftn(phi)
    d0 = np.real(ifftn(1j*KX * phik)).astype(dtype)  # ∂x
    d1 = np.real(ifftn(1j*KY * phik)).astype(dtype)  # ∂y
    d2 = np.real(ifftn(1j*KZ * phik)).astype(dtype)  # ∂z
    return d0, d1, d2

# First derivatives of a spatial 3x3 tensor field T_{ij}(x)
# T shape: (3,3,N,N,N) -> dT shape: (3,3,3,N,N,N), leading axis = derivative direction
def d_tensor(T):
    dT = np.zeros((3,3,3,N,N,N), dtype=dtype)
    for a in range(3):
        for b in range(3):
            Tk = fftn(T[a,b])
            dT[0,a,b] = np.real(ifftn(1j*KX * Tk)).astype(dtype)  # ∂x
            dT[1,a,b] = np.real(ifftn(1j*KY * Tk)).astype(dtype)  # ∂y
            dT[2,a,b] = np.real(ifftn(1j*KZ * Tk)).astype(dtype)  # ∂z
    return dT

# Divergence of a scalar field's gradient (Laplacian) via spectral
def laplacian(phi):
    phik = fftn(phi)
    lap = np.real(ifftn(-(KX**2 + KY**2 + KZ**2) * phik)).astype(dtype)
    return lap

# Divergence of a vector field V_i (sum ∂_i V_i)
def div_vector(V):
    res = np.zeros_like(V[0])
    for i,Ki in enumerate([KX, KY, KZ]):
        Vk = fftn(V[i])
        res += np.real(ifftn(1j*Ki * Vk)).astype(dtype)
    return res

# ----------------------------
# Induced metric from Φ (starter ansatz; replace with your HES‑P₀ metric)
# ----------------------------
sigma = 0.20  # smooth core to reduce high‑k
Phi = np.exp(-((X-0.5)**2 + (Y-0.5)**2 + (Z-0.5)**2) / (sigma**2)).astype(dtype)

# Weak‑field static ansatz: g_00 = -1 + 2Φ, g_ij = (1 + 2Φ) δ_ij, g_0i = 0
g00 = -1.0 + 2.0 * Phi
g0i = [np.zeros_like(Phi), np.zeros_like(Phi), np.zeros_like(Phi)]
gij = np.zeros((3,3,N,N,N), dtype=dtype)
for i in range(3):
    gij[i,i] = 1.0 + 2.0 * Phi  # diagonal spatial metric

# Inverse metric (linearized; for stronger fields do per‑voxel 3x3 inversion)
g00_inv = -1.0 - 2.0 * Phi
gij_inv = np.zeros_like(gij)
for i in range(3):
    gij_inv[i,i] = 1.0 - 2.0 * Phi

# ----------------------------
# Spatial Christoffel symbols Γ^i_{jk}
# ----------------------------
# Γ^i_{jk} = 1/2 g^{iℓ} (∂_j g_{ℓk} + ∂_k g_{ℓj} - ∂_ℓ g_{jk}), spatial indices only
dg = d_tensor(gij)  # shape (3,3,3,N,N,N), leading derivative axis

Gamma = np.zeros((3,3,3,N,N,N), dtype=dtype)  # Γ^i_{jk}
for i in range(3):
    for j in range(3):
        for k in range(3):
            s = np.zeros_like(Phi)
            for ell in range(3):
                term = dg[j,ell,k] + dg[k,ell,j] - dg[ell,j,k]
                s += gij_inv[i,ell] * term
            Gamma[i,j,k] = 0.5 * s

# ----------------------------
# Spatial Ricci tensor R_{ij}
# ----------------------------
# R_{ij} = ∂_k Γ^k_{ij} - ∂_j Γ^k_{ik} + Γ^k_{ij} Γ^ℓ_{kℓ} - Γ^ℓ_{ik} Γ^k_{jℓ}
# Derivatives of Γ
dGamma = np.zeros((3,3,3,3,N,N,N), dtype=dtype)  # leading derivative axis
for i in range(3):
    for j in range(3):
        for k in range(3):
            Gk = fftn(Gamma[i,j,k])
            dGamma[0,i,j,k] = np.real(ifftn(1j*KX * Gk)).astype(dtype)  # ∂x
            dGamma[1,i,j,k] = np.real(ifftn(1j*KY * Gk)).astype(dtype)  # ∂y
            dGamma[2,i,j,k] = np.real(ifftn(1j*KZ * Gk)).astype(dtype)  # ∂z

# Traces needed: Γ^ℓ_{kℓ} and Γ^k_{ik}
Gamma_trace_k = np.zeros((3,N,N,N), dtype=dtype)  # for each k, sum over ℓ: Γ^ℓ_{kℓ}
Gamma_trace_i = np.zeros((3,N,N,N), dtype=dtype)  # for each i, sum over k: Γ^k_{ik}
for k in range(3):
    # sum over ℓ = j index of Γ^ℓ_{kℓ} → Γ[ell,k,ell]
    tr = np.zeros_like(Phi)
    for ell in range(3):
        tr += Gamma[ell,k,ell]
    Gamma_trace_k[k] = tr
for i in range(3):
    tr = np.zeros_like(Phi)
    for k in range(3):
        tr += Gamma[k,i,k]
    Gamma_trace_i[i] = tr

# Build Rij
Rij = np.zeros((3,3,N,N,N), dtype=dtype)
for i in range(3):
    for j in range(3):
        # ∂_k Γ^k_{ij} = sum_k ∂_k Γ^k_{ij}
        term1 = dGamma[0,0,j,i] + dGamma[1,1,j,i] + dGamma[2,2,j,i]
        # ∂_j Γ^k_{ik} = derivative along j of trace Γ^k_{ik}
        Gti_k = Gamma_trace_i[i]
        Kj = [KX, KY, KZ][j]
        term2 = np.real(ifftn(1j*Kj * fftn(Gti_k))).astype(dtype)
        # Γ^k_{ij} Γ^ℓ_{kℓ}
        term3 = np.zeros_like(Phi)
        for k in range(3):
            term3 += Gamma[k,i,j] * Gamma_trace_k[k]
        # Γ^ℓ_{ik} Γ^k_{jℓ}
        term4 = np.zeros_like(Phi)
        for k in range(3):
            for ell in range(3):
                term4 += Gamma[ell,i,k] * Gamma[k,j,ell]
        Rij[i,j] = term1 - term2 + term3 - term4

# ----------------------------
# Ricci scalar R and Einstein tensor G_{ij}
# ----------------------------
R = np.zeros_like(Phi)
for i in range(3):
    R += gij_inv[i,i] * Rij[i,i]

Gij = np.zeros_like(Rij)
for i in range(3):
    for j in range(3):
        Gij[i,j] = Rij[i,j] - 0.5 * gij[i,j] * R

# ----------------------------
# Temporal components (static weak‑field closure)
# ----------------------------
# In static weak‑field: G00 ≈ 2 ∇²Φ, G0i ≈ 0
G00 = 2.0 * laplacian(Phi)
G0i_vals = [np.zeros_like(Phi), np.zeros_like(Phi), np.zeros_like(Phi)]

# ----------------------------
# Effective stress–energy T_{μν} from Φ (proxy; swap in HES‑P₀ mapping)
# ----------------------------
dPhix, dPhiy, dPhiz = d_scalar(Phi)
grad_sq = (gij_inv[0,0] * dPhix**2) + (gij_inv[1,1] * dPhiy**2) + (gij_inv[2,2] * dPhiz**2)

kappa = 0.0  # set to your vacuum-pressure parameter when mapping
T00 = -0.5 * g00 * grad_sq + kappa * (Phi**4) * g00
T0i = [np.zeros_like(Phi), np.zeros_like(Phi), np.zeros_like(Phi)]
Tij = np.zeros_like(gij)
# Diagonals
Tij[0,0] = dPhix*dPhix - 0.5 * gij[0,0] * grad_sq + kappa * (Phi**4) * gij[0,0]
Tij[1,1] = dPhiy*dPhiy - 0.5 * gij[1,1] * grad_sq + kappa * (Phi**4) * gij[1,1]
Tij[2,2] = dPhiz*dPhiz - 0.5 * gij[2,2] * grad_sq + kappa * (Phi**4) * gij[2,2]
# Off‑diagonals (symmetric)
Tij[0,1] = dPhix * dPhiy; Tij[1,0] = Tij[0,1]
Tij[0,2] = dPhix * dPhiz; Tij[2,0] = Tij[0,2]
Tij[1,2] = dPhiy * dPhiz; Tij[2,1] = Tij[1,2]

# ----------------------------
# Residuals (dimensionless normalization: 8π coupling)
# ----------------------------
coeff = 8.0 * np.pi

res_G00 = norm(G00 - coeff * T00)
res_G0 = [norm(G0i_vals[i] - coeff * T0i[i]) for i in range(3)]

res_Gij_diag = [norm(Gij[i,i] - coeff * Tij[i,i]) for i in range(3)]
res_Gij_off  = [norm(Gij[0,1] - coeff * Tij[0,1]),
                norm(Gij[0,2] - coeff * Tij[0,2]),
                norm(Gij[1,2] - coeff * Tij[1,2])]

print("Residual norms:")
print(f"  G00 vs 8πT00: {res_G00:.3e}")
print(f"  G0i vs 8πT0i: {res_G0[0]:.3e}, {res_G0[1]:.3e}, {res_G0[2]:.3e}")
print(f"  Gij vs 8πTij (diag): {res_Gij_diag[0]:.3e}, {res_Gij_diag[1]:.3e}, {res_Gij_diag[2]:.3e}")
print(f"  Gij vs 8πTij (off):  {res_Gij_off[0]:.3e}, {res_Gij_off[1]:.3e}, {res_Gij_off[2]:.3e}")

# ----------------------------
# Conservation and Bianchi proxies (static)
# ----------------------------
# ∇_μ T^{μν} ~ 0 in static smooth case; here we check spatial divergence of flux components ~ 0
div_T0 = 0.0  # static, no flux

# Bianchi proxy: spatial contraction ∇^i G_{ij} ~ 0
def div_G_column(j):
    s = np.zeros_like(Phi)
    for i in range(3):
        Gij_k = fftn(Gij[i,j])
        Ki = [KX, KY, KZ][i]
        s += np.real(ifftn(1j*Ki * Gij_k)).astype(dtype)
    return s

bianchi_norms = [norm(div_G_column(j)) for j in range(3)]
print(f"  div(T^0) proxy: {div_T0:.1e}")
print(f"  Bianchi (spatial divergence norms): {bianchi_norms[0]:.3e}, {bianchi_norms[1]:.3e}, {bianchi_norms[2]:.3e}")


Residual norms:
  G00 vs 8πT00: 5.684e-02
  G0i vs 8πT0i: 0.000e+00, 0.000e+00, 0.000e+00
  Gij vs 8πTij (diag): 1.241e-01, 1.241e-01, 1.241e-01
  Gij vs 8πTij (off):  4.337e-02, 4.337e-02, 4.337e-02
  div(T^0) proxy: 0.0e+00
  Bianchi (spatial divergence norms): 1.051e+00, 1.051e+00, 1.051e+00


In [ ]:
# HES-P₀ v22.5 — Full Metric Pipeline: g -> Γ -> R -> G (spectral, periodic, corrected)
# Authors: Chris & Copilot

import numpy as np

# ----------------------------
# Grid and spectral setup
# ----------------------------
N = 64         # Increase to 128–256 for tighter residuals
L = 1.0
dx = L / N
dtype = np.float64

x = np.linspace(0, L, N, endpoint=False, dtype=dtype)
X, Y, Z = np.meshgrid(x, x, x, indexing='ij')

kx = 2.0 * np.pi * np.fft.fftfreq(N, d=dx)
ky = 2.0 * np.pi * np.fft.fftfreq(N, d=dx)
kz = 2.0 * np.pi * np.fft.fftfreq(N, d=dx)
KX, KY, KZ = np.meshgrid(kx, ky, kz, indexing='ij')

def fftn(a): return np.fft.fftn(a)
def ifftn(a): return np.fft.ifftn(a)
def norm(field): return np.linalg.norm(field) / field.size

# ----------------------------
# Spectral derivative helpers
# ----------------------------
def d_scalar(phi):
    phik = fftn(phi)
    d0 = np.real(ifftn(1j*KX * phik)).astype(dtype)
    d1 = np.real(ifftn(1j*KY * phik)).astype(dtype)
    d2 = np.real(ifftn(1j*KZ * phik)).astype(dtype)
    return d0, d1, d2

def d_tensor(T):
    # T: (3,3,N,N,N) -> (3,3,3,N,N,N)
    dT = np.zeros((3,3,3,N,N,N), dtype=dtype)
    for a in range(3):
        for b in range(3):
            Tk = fftn(T[a,b])
            dT[0,a,b] = np.real(ifftn(1j*KX * Tk)).astype(dtype)
            dT[1,a,b] = np.real(ifftn(1j*KY * Tk)).astype(dtype)
            dT[2,a,b] = np.real(ifftn(1j*KZ * Tk)).astype(dtype)
    return dT

def laplacian(phi):
    phik = fftn(phi)
    return np.real(ifftn(-(KX**2 + KY**2 + KZ**2) * phik)).astype(dtype)

# ----------------------------
# Echo field and induced metric
# ----------------------------
sigma = 0.30  # smoother to reduce high-k; tighten residuals
Phi = np.exp(-((X-0.5)**2 + (Y-0.5)**2 + (Z-0.5)**2) / (sigma**2)).astype(dtype)

# Spatial metric (diagonal starter; swap with your induced metric when ready)
gij = np.zeros((3,3,N,N,N), dtype=dtype)
for i in range(3):
    gij[i,i] = 1.0 + 2.0 * Phi

# Per-voxel 3x3 inverse of gij
G = np.stack([np.stack([gij[i,j] for j in range(3)], axis=-1) for i in range(3)], axis=-2)  # (...,3,3)
G_flat = G.reshape(-1, 3, 3)
Ginv_flat = np.linalg.inv(G_flat)
Ginv = Ginv_flat.reshape(N, N, N, 3, 3)
gij_inv = np.zeros_like(gij)
for i in range(3):
    for j in range(3):
        gij_inv[i,j] = Ginv[..., i, j]

# ----------------------------
# Christoffels Γ^i_{jk}
# ----------------------------
dg = d_tensor(gij)  # (deriv, i, j, grid)

Gamma = np.zeros((3,3,3,N,N,N), dtype=dtype)
for i in range(3):
    for j in range(3):
        for k in range(3):
            s = np.zeros_like(Phi)
            for ell in range(3):
                term = dg[j,ell,k] + dg[k,ell,j] - dg[ell,j,k]
                s += gij_inv[i,ell] * term
            Gamma[i,j,k] = 0.5 * s

# ----------------------------
# Ricci tensor R_{ij}
# ----------------------------
dGamma = np.zeros((3,3,3,3,N,N,N), dtype=dtype)
for i in range(3):
    for j in range(3):
        for k in range(3):
            Gk = fftn(Gamma[i,j,k])
            dGamma[0,i,j,k] = np.real(ifftn(1j*KX * Gk)).astype(dtype)
            dGamma[1,i,j,k] = np.real(ifftn(1j*KY * Gk)).astype(dtype)
            dGamma[2,i,j,k] = np.real(ifftn(1j*KZ * Gk)).astype(dtype)

Gamma_trace_k = np.zeros((3,N,N,N), dtype=dtype)
Gamma_trace_i = np.zeros((3,N,N,N), dtype=dtype)
for k in range(3):
    tr = np.zeros_like(Phi)
    for ell in range(3):
        tr += Gamma[ell,k,ell]
    Gamma_trace_k[k] = tr
for i in range(3):
    tr = np.zeros_like(Phi)
    for k in range(3):
        tr += Gamma[k,i,k]
    Gamma_trace_i[i] = tr

Rij = np.zeros((3,3,N,N,N), dtype=dtype)
for i in range(3):
    for j in range(3):
        term1 = dGamma[0,0,j,i] + dGamma[1,1,j,i] + dGamma[2,2,j,i]
        Kj = [KX, KY, KZ][j]
        term2 = np.real(ifftn(1j*Kj * fftn(Gamma_trace_i[i]))).astype(dtype)
        term3 = np.zeros_like(Phi)
        for k in range(3):
            term3 += Gamma[k,i,j] * Gamma_trace_k[k]
        term4 = np.zeros_like(Phi)
        for k in range(3):
            for ell in range(3):
                term4 += Gamma[ell,i,k] * Gamma[k,j,ell]
        Rij[i,j] = term1 - term2 + term3 - term4

# ----------------------------
# Ricci scalar and Einstein tensors
# ----------------------------
R = np.zeros_like(Phi)
for i in range(3):
    for j in range(3):
        R += gij_inv[i,j] * Rij[i,j]

Gij_lower = np.zeros_like(Rij)
for i in range(3):
    for j in range(3):
        Gij_lower[i,j] = Rij[i,j] - 0.5 * gij[i,j] * R

G_mixed = np.zeros_like(Rij)
for i in range(3):
    for j in range(3):
        s = np.zeros_like(Phi)
        for ell in range(3):
            s += gij_inv[i,ell] * Gij_lower[ell,j]
        G_mixed[i,j] = s

# ----------------------------
# Temporal weak-field closure
# ----------------------------
G00 = 2.0 * laplacian(Phi)

# ----------------------------
# Scalar-field T_{μν} proxy
# ----------------------------
dPhix, dPhiy, dPhiz = d_scalar(Phi)
T_lower = np.zeros_like(gij)
T_lower[0,0] = dPhix*dPhix
T_lower[1,1] = dPhiy*dPhiy
T_lower[2,2] = dPhiz*dPhiz
T_lower[0,1] = dPhix*dPhiy; T_lower[1,0] = T_lower[0,1]
T_lower[0,2] = dPhix*dPhiz; T_lower[2,0] = T_lower[0,2]
T_lower[1,2] = dPhiy*dPhiz; T_lower[2,1] = T_lower[1,2]

T_mixed = np.zeros_like(T_lower)
for i in range(3):
    for j in range(3):
        s = np.zeros_like(Phi)
        for ell in range(3):
            s += gij_inv[i,ell] * T_lower[ell,j]
        T_mixed[i,j] = s

T00 = T_lower[0,0] + T_lower[1,1] + T_lower[2,2]

# ----------------------------
# Residuals
# ----------------------------
coeff = 8.0 * np.pi

res_G00 = norm(G00 - coeff * T00)
res_G_mixed_diag = [norm(G_mixed[i,i] - coeff * T_mixed[i,i]) for i in range(3)]
res_G_mixed_off  = [norm(G_mixed[0,1] - coeff * T_mixed[0,1]),
                    norm(G_mixed[0,2] - coeff * T_mixed[0,2]),
                    norm(G_mixed[1,2] - coeff * T_mixed[1,2])]

print("Residual norms (mixed indices):")
print(f"  G00 vs 8πT00: {res_G00:.3e}")
print(f"  G^i_i vs 8πT^i_i: {res_G_mixed_diag[0]:.3e}, {res_G_mixed_diag[1]:.3e}, {res_G_mixed_diag[2]:.3e}")
print(f"  G^i_j vs 8πT^i_j (off): {res_G_mixed_off[0]:.3e}, {res_G_mixed_off[1]:.3e}, {res_G_mixed_off[2]:.3e}")

# ----------------------------
# Covariant divergence ∇_i G^i{}_j (Bianchi)
# ----------------------------
def d_i(field):  # returns [∂x, ∂y, ∂z] of scalar field
    fk = fftn(field)
    return [np.real(ifftn(1j*KX * fk)).astype(dtype),
            np.real(ifftn(1j*KY * fk)).astype(dtype),
            np.real(ifftn(1j*KZ * fk)).astype(dtype)]

div_cov = [np.zeros_like(Phi) for _ in range(3)]
for j in range(3):
    s = np.zeros_like(Phi)
    for i in range(3):
        # ∂_i G^i{}_j
        dGij_i = d_i(G_mixed[i,j])[i]
        s += dGij_i
        # + Γ^i_{iℓ} G^ℓ{}_j
        add1 = np.zeros_like(Phi)
        for ell in range(3):
            add1 += Gamma[i,i,ell] * G_mixed[ell,j]
        s += add1
        # - Γ^ℓ_{ij} G^i{}_ℓ
        sub1 = np.zeros_like(Phi)
        for ell in range(3):
            sub1 += Gamma[ell,i,j] * G_mixed[i,ell]
        s -= sub1
    div_cov[j] = s

bianchi_cov_norms = [norm(div_cov[j]) for j in range(3)]
print(f"  Bianchi (covariant divergence norms): {bianchi_cov_norms[0]:.3e}, {bianchi_cov_norms[1]:.3e}, {bianchi_cov_norms[2]:.3e}")


Residual norms (mixed indices):
  G00 vs 8πT00: 1.777e-01
  G^i_i vs 8πT^i_i: 4.074e-02, 4.074e-02, 4.074e-02
  G^i_j vs 8πT^i_j (off): 2.167e-02, 2.167e-02, 2.167e-02
  Bianchi (covariant divergence norms): 8.945e-03, 8.945e-03, 8.945e-03
